# Hyperparameter Tuning & Ensemble Optimization

This notebook implements three hyperparameter tuning techniques (Grid Search, Randomized Search, and Successive Halving) for LightGBM, and trains a weighted ensemble of LightGBM, XGBoost, and CatBoost with Repeated Stratified K-Fold cross-validation.

### Fixed Syntactic & Mathematical Errors from original script:
- Missing colons on `for` loops.
- Missing math operators for BMI, Agility Ratio, Strength Index, Speed Score, Explosiveness, Power Index, Catch Radius, and Height Adjusted Speed.
- Unquoted strings for filenames, columns, and target label.
- Fixed missing multiplication operators and array slice symbols in ensemble prediction calculation.

In [1]:
# ==========================================
# 1. LOAD IMPORTS & SETUP
# ==========================================
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.model_selection import RepeatedStratifiedKFold

# Hyperparameter tuning libraries
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.experimental import enable_halving_search_cv  # Required to enable successive halving
from sklearn.model_selection import HalvingGridSearchCV, HalvingRandomSearchCV

# Classifiers
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

import matplotlib.pyplot as plt
%matplotlib inline

ModuleNotFoundError: No module named 'sklearn'

In [ ]:
# ==========================================
# 2. LOAD DATA
# ==========================================
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
sample_sub = pd.read_csv('sample_submission.csv')

In [ ]:
# ==========================================
# 3. ADVANCED FEATURE ENGINEERING
# ==========================================
for df in [train, test]:
    # Body Composition (Weight / Height squared)
    df['BMI'] = df['Weight'] / (df['Height'] ** 2)
    
    # Agility ratio (Agility 3-cone / Shuttle)
    df['Agility_Ratio'] = df['Agility_3cone'] / df['Shuttle']

    # Strength per unit weight
    df['Strength_Index'] = df['Bench_Press_Reps'] / df['Weight']

    # Speed relative to position type
    df['Speed_pos_type_diff'] = df['Sprint_40yd'] - df.groupby('Position_Type')['Sprint_40yd'].transform('mean')
    
    # Size-Adjusted Athleticism
    df['Speed_Score'] = df['Weight'] * df['Sprint_40yd']
    
    # Power & Explosiveness
    df['Explosiveness'] = df['Vertical_Jump'] + df['Broad_Jump'] 
    df['Power_Index'] = df['Vertical_Jump'] * df['Weight']
    
    # Directional Change
    df['Total_Agility'] = df['Agility_3cone'] + df['Shuttle']
    
    # Missing Value Indicators
    for col in ['Agility_3cone', 'Shuttle', 'Bench_Press_Reps']:
        df[f'{col}_missing'] = df[col].isnull().astype(int)
    
    # Composite Metrics
    df['Catch_Radius'] = df['Height'] + (df['Vertical_Jump'] / 100) + (df['Broad_Jump'] / 100) - (df['Shuttle'] / 10)
    df['Height_Adj_Speed'] = (df['Weight'] / (df['Height'] ** 0.5)) / (df['Sprint_40yd'] ** 4)
    
    # Position-Relative Stats
    for stat in ['Sprint_40yd', 'BMI', 'Catch_Radius']:
        df[f'{stat}_pos_diff'] = df[stat] - df.groupby('Position')[stat].transform('mean')

In [ ]:
# ==========================================
# 4. PREPROCESSING & IMPUTATION
# ==========================================
# Handle 'School' using Frequency Encoding
school_freq = train['School'].value_counts().to_dict()
train['School_Freq'] = train['School'].map(school_freq)
test['School_Freq'] = test['School'].map(school_freq).fillna(0)

# Label Encoding for remaining categoricals
le = LabelEncoder()
for col in ['Player_Type', 'Position_Type', 'Position']:
    train[col] = le.fit_transform(train[col].astype(str))
    test[col] = le.transform(test[col].astype(str))

# Setup Features (X) and Target (y)
X = train.drop(columns=['Drafted'])
y = train['Drafted']

# Drop high-cardinality/unnecessary text and key columns
X = X.drop(columns=['Id', 'School'])
test_cleaned = test.drop(columns=['Id', 'School'])

# Advanced Imputation using IterativeImputer (MICE)
imputer = IterativeImputer(random_state=2025)
X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)
test_imputed = pd.DataFrame(imputer.transform(test_cleaned), columns=test_cleaned.columns)

## 5. HYPERPARAMETER TUNING IMPLEMENTATIONS

In [ ]:
## Method 1: Grid Search (GridSearchCV)
# Exhaustively searches over a specified parameter grid.
# Good for small parameter grids.

lgbm = LGBMClassifier(random_state=2025, objective='binary', verbose=-1)

grid_parameters = {
    'n_estimators': [100, 150],
    'learning_rate': [0.02, 0.05],
    'num_leaves': [15, 25],
}

grid_search = GridSearchCV(
    estimator=lgbm,
    param_grid=grid_parameters,
    scoring='roc_auc',
    cv=3,
    n_jobs=-1,
    verbose=1
)

# Fit the grid search to tune hyperparameters
grid_search.fit(X_imputed, y)

print(f"Best Parameters (Grid Search): {grid_search.best_params_}")
print(f"Best AUC Score (Grid Search): {grid_search.best_score_:.4f}")

In [ ]:
## Method 2: Randomized Search (RandomizedSearchCV)
# Randomly samples combinations of parameters from defined distributions.
# Ideal for larger parameter spaces.

param_distributions = {
    'n_estimators': [100, 145, 200],
    'learning_rate': [0.01, 0.02, 0.05, 0.1],
    'num_leaves': [20, 25, 31, 40],
    'lambda_l2': [0.0, 1.0, 5.0],
}

random_search = RandomizedSearchCV(
    estimator=lgbm,
    param_distributions=param_distributions,
    n_iter=8,  # Number of random configurations to test
    scoring='roc_auc',
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

# Fit the randomized search
random_search.fit(X_imputed, y)

print(f"Best Parameters (Randomized Search): {random_search.best_params_}")
print(f"Best AUC Score (Randomized Search): {random_search.best_score_:.4f}")

In [ ]:
## Method 3: Successive Halving (HalvingGridSearchCV)
# Trains all candidates on a tiny subset of resources (samples),
# then iteratively selects the best performers to train on more resources.
# Extremely fast and resource-efficient.

halving_grid = {
    'n_estimators': [50, 100, 150],
    'learning_rate': [0.01, 0.02, 0.05],
    'num_leaves': [15, 25, 31],
}

halving_search = HalvingGridSearchCV(
    estimator=lgbm,
    param_grid=halving_grid,
    scoring='roc_auc',
    cv=3,
    factor=3,  # Candidate reduction factor in each round
    min_resources='exhaust',  # Automatically determine starting resources
    n_jobs=-1,
    verbose=1,
    random_state=42
)

# Fit the successive halving search
halving_search.fit(X_imputed, y)

print(f"Best Parameters (Successive Halving): {halving_search.best_params_}")
print(f"Best AUC Score (Successive Halving): {halving_search.best_score_:.4f}")

## 6. CROSS-VALIDATION ENSEMBLE WITH TUNED HYPERPARAMETERS

We update the LightGBM classifier with the optimal parameters found via **Successive Halving** (or your preferred method) and run the full cross-validation ensemble.

In [ ]:
# Set base parameters and update with the tuned hyperparameters
tuned_lgbm_params = {
    'n_estimators': 145,
    'learning_rate': 0.02,
    'random_state': 2025,
    'objective': 'binary',
    'num_leaves': 25,
    'lambda_l2': 1.0,
    'is_unbalance': True,
    'verbose': -1
}

# Update parameters with tuned results if desired (e.g. from Successive Halving)
best_params = halving_search.best_params_
tuned_lgbm_params.update(best_params)

# Model definition
cat_model = CatBoostClassifier(
    iterations=150, 
    depth=6,
    learning_rate=0.02,
    random_seed=2025, 
    early_stopping_rounds=20,
    verbose=0,
    subsample=0.8,
    bootstrap_type='Bernoulli',
    auto_class_weights='Balanced',  # Handle class imbalance
    l2_leaf_reg=3.0,  # L2 regularization
)

xgb_model = XGBClassifier(
    n_estimators=145,
    max_depth=5,
    learning_rate=0.02,
    random_state=2025, 
    eval_metric='logloss',
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,  # L2 regularization
    reg_alpha=0.1,   # L1 regularization
)

lgbm_model = LGBMClassifier(**tuned_lgbm_params)

# K-Fold definition (5 splits * 5 repeats = 25 total folds)
rskf = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=42)

ensemble_preds = np.zeros(len(test_imputed))

for fold, (train_idx, valid_idx) in enumerate(rskf.split(X_imputed, y)):
    X_t, X_v = X_imputed.iloc[train_idx], X_imputed.iloc[valid_idx]
    y_t, y_v = y.iloc[train_idx], y.iloc[valid_idx]
    
    # Train individual models
    cat_model.fit(X_t, y_t)
    xgb_model.fit(X_t, y_t)
    lgbm_model.fit(X_t, y_t)
    
    # Balanced weighted ensemble predictions
    fold_preds = (
        0.4 * cat_model.predict_proba(test_imputed)[:, 1] + 
        0.3 * xgb_model.predict_proba(test_imputed)[:, 1] + 
        0.3 * lgbm_model.predict_proba(test_imputed)[:, 1]
    )
    # Correctly divide by total folds (5 splits * 5 repeats = 25)
    ensemble_preds += fold_preds / (5 * 5)

# Create submission file
submission = sample_sub.copy()
submission['Drafted'] = ensemble_preds
submission.to_csv('submission_final_improved.csv', index=False)
print("Submission file created. Expected improvement over baseline AUC.")

## 7. CATBOOST OVERFITTING VISUALIZATION (ONE FOLD)

In [ ]:
# Use ONE fold to visualize overfitting
train_idx, valid_idx = list(rskf.split(X_imputed, y))[0]
X_t, X_v = X_imputed.iloc[train_idx], X_imputed.iloc[valid_idx]
y_t, y_v = y.iloc[train_idx], y.iloc[valid_idx]

cat_model_check = CatBoostClassifier(
    iterations=150, 
    depth=6,
    learning_rate=0.02,
    random_seed=2025, 
    early_stopping_rounds=20,
    verbose=0,
    subsample=0.8,
    bootstrap_type='Bernoulli',
    auto_class_weights='Balanced',
    l2_leaf_reg=3.0,
)

cat_model_check.fit(
    X_t, y_t,
    eval_set=(X_v, y_v),  # Pass validation set
    verbose=True
)

# Plot the logloss curves
train_loss = cat_model_check.evals_result_['learn']['Logloss']
val_loss = cat_model_check.evals_result_['validation']['Logloss']

plt.figure(figsize=(10, 5))
plt.plot(train_loss, label='Train Logloss')
plt.plot(val_loss, label='Val Logloss')
plt.xlabel('Iterations')
plt.ylabel('Logloss')
plt.title('CatBoost - Overfitting Check')
plt.legend()
plt.savefig('logloss_curve.png')
plt.show()